In [1]:
!git clone https://github.com/aliakseizvertouski/olist.git

Cloning into 'olist'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 67 (delta 33), reused 6 (delta 3), pack-reused 14 (from 1)
Receiving objects: 100% (67/67), 42.62 MiB | 17.24 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv('/content/olist/olist_customers_dataset.csv')
geo = pd.read_csv('/content/olist/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/olist/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/olist/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/olist/olist_orders_dataset.csv')
products = pd.read_csv('/content/olist/olist_products_dataset.csv')
sellers = pd.read_csv('/content/olist/olist_sellers_dataset.csv')

# Информация о датасете

In [4]:
print (customers.columns)
print (geo.columns)
print (order_items.columns)
print (order_payments.columns)
print (order_reviews.columns)
print (orders.columns)
print (products.columns)
print (sellers.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['product_id', 'prod

# Анализ продаж


In [5]:
df_prices = order_items[['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']].merge(
    orders[['order_id', 'order_purchase_timestamp']],
    on='order_id'
)

df_prices['purchase_date'] = pd.to_datetime(df_prices['order_purchase_timestamp'])
df_prices = df_prices.sort_values(by=['seller_id', 'product_id', 'purchase_date'])

df_prices['prev_price'] = df_prices.groupby(['seller_id', 'product_id'])['price'].shift(1)
df_prices['discount_percent'] = ((df_prices['prev_price'] - df_prices['price']) / df_prices['prev_price']) * 100
df_prices['discount_percent'] = df_prices['discount_percent'].fillna(0)

In [7]:


df_prices['purchase_date'] = pd.to_datetime(df_prices['order_purchase_timestamp'])
df_prices = df_prices.sort_values(by=['seller_id', 'product_id', 'purchase_date'])


df_prices['prev_price'] = df_prices.groupby(['seller_id', 'product_id'])['price'].shift(1)
df_prices['discount_percent'] = ((df_prices['prev_price'] - df_prices['price']) / df_prices['prev_price']) * 100
df_prices['discount_percent'] = df_prices['discount_percent'].fillna(0)


def f(arr):
    rank = 0
    # Превращаем в список, если пришла серия, чтобы работать через enumerate
    values = arr.tolist()
    result = [0] * len(values)
    for idx, value in enumerate(values):
        if value != 0:
            rank += 1
        result[idx] = rank
    return pd.Series(result, index=arr.index)

df_prices['price_period_rank'] = df_prices.groupby(['seller_id', 'product_id'])['discount_percent'].transform(f)

df_prices


,order_id,order_item_id,product_id,seller_id,price,order_purchase_timestamp,purchase_date,prev_price,discount_percent,price_period_rank
93696,d455a8cb295653b55abda06d434ab492,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-09-26 22:17:05,2017-09-26 22:17:05,NaN,0.0,0
69082,9dc8d1a6f16f1b89874c29c9d8d30447,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-10-12 13:33:22,2017-10-12 13:33:22,895.0,0.0,0
55943,7f39ba4c9052be115350065d07583cac,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-10-18 08:16:34,2017-10-18 08:16:34,895.0,0.0,0
57028,81bee7ecbeae8e9b41bef6d41b146b12,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,69.90,2017-03-15 22:26:48,2017-03-15 22:26:48,NaN,0.0,0
57029,81bee7ecbeae8e9b41bef6d41b146b12,2,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,69.90,2017-03-15 22:26:48,2017-03-15 22:26:48,69.9,0.0,0
...,...,...,...,...,...,...,...,...,...,...
42778,616b813dbea8acc9de0ca0380cd89b83,1,dbd024d4182504993ad1e3cd2ee9d9e9,ffff564a4f9085cd26170f4732393726,29.40,2017-01-16 19:04:32,2017-01-16 19:04:32,NaN,0.0,0
98461,df537c849af44beef86a7ef7de12126a,1,dbd024d4182504993ad1e3cd2ee9d9e9,ffff564a4f9085cd26170f4732393726,29.40,2017-01-19 21:48:41,2017-01-19 21:48:41,29.4,0.0,0
20859,2fbb05b3ee700e1897b9fa501e416005,1,de6517dda8e49774f58c07f80abc8d7a,ffff564a4f9085cd26170f4732393726,69.00,2016-10-10 15:37:12,2016-10-10 15:37:12,NaN,0.0,0
34958,4f21594649a0235ad65ded11ba7c8ea7,1,e20b58fe57d487f33247e6cc1154eb9c,ffff564a4f9085cd26170f4732393726,103.95,2017-04-03 22:08:47,2017-04-03 22:08:47,NaN,0.0,0


In [13]:
df_prices['purchase_date'] = pd.to_datetime(df_prices['order_purchase_timestamp'])
df_prices = df_prices.sort_values(by=['seller_id', 'product_id', 'purchase_date'])


df_prices['prev_price'] = df_prices.groupby(['seller_id', 'product_id'])['price'].shift(1)
df_prices['discount_percent'] = ((df_prices['prev_price'] - df_prices['price']) / df_prices['prev_price']) * 100
df_prices['discount_percent'] = df_prices['discount_percent'].fillna(0)


def f(arr):
    rank = 0
    values = arr.tolist()
    result = [0] * len(values)
    for idx, value in enumerate(values):
        if idx > 0 and value != values[idx - 1]:
            rank += 1
        result[idx] = rank
    return pd.Series(result, index=arr.index)

df_prices['price_period_rank'] = df_prices.groupby(['seller_id', 'product_id'])['price'].transform(f)

df_prices['price_period_size'] = df_prices.groupby(['seller_id', 'product_id', 'price_period_rank'])['price_period_rank'].transform('size')

df_prices


,order_id,order_item_id,product_id,seller_id,price,order_purchase_timestamp,purchase_date,prev_price,discount_percent,price_period_rank,price_period_size
93696,d455a8cb295653b55abda06d434ab492,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-09-26 22:17:05,2017-09-26 22:17:05,NaN,0.0,0,3
69082,9dc8d1a6f16f1b89874c29c9d8d30447,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-10-12 13:33:22,2017-10-12 13:33:22,895.0,0.0,0,3
55943,7f39ba4c9052be115350065d07583cac,1,a2ff5a97bf95719e38ea2e3b4105bce8,0015a82c2db000af6aaaf3ae2ecb0532,895.00,2017-10-18 08:16:34,2017-10-18 08:16:34,895.0,0.0,0,3
57028,81bee7ecbeae8e9b41bef6d41b146b12,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,69.90,2017-03-15 22:26:48,2017-03-15 22:26:48,NaN,0.0,0,7
57029,81bee7ecbeae8e9b41bef6d41b146b12,2,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,69.90,2017-03-15 22:26:48,2017-03-15 22:26:48,69.9,0.0,0,7
...,...,...,...,...,...,...,...,...,...,...,...
42778,616b813dbea8acc9de0ca0380cd89b83,1,dbd024d4182504993ad1e3cd2ee9d9e9,ffff564a4f9085cd26170f4732393726,29.40,2017-01-16 19:04:32,2017-01-16 19:04:32,NaN,0.0,0,2
98461,df537c849af44beef86a7ef7de12126a,1,dbd024d4182504993ad1e3cd2ee9d9e9,ffff564a4f9085cd26170f4732393726,29.40,2017-01-19 21:48:41,2017-01-19 21:48:41,29.4,0.0,0,2
20859,2fbb05b3ee700e1897b9fa501e416005,1,de6517dda8e49774f58c07f80abc8d7a,ffff564a4f9085cd26170f4732393726,69.00,2016-10-10 15:37:12,2016-10-10 15:37:12,NaN,0.0,0,1
34958,4f21594649a0235ad65ded11ba7c8ea7,1,e20b58fe57d487f33247e6cc1154eb9c,ffff564a4f9085cd26170f4732393726,103.95,2017-04-03 22:08:47,2017-04-03 22:08:47,NaN,0.0,0,1


In [ ]:
target_seller = '001cca7ae9ae17fb1caed9dfb1094831'
target_product = '08574b074924071f4e201e151b152b4e'

target_seller_data = df_prices[df_prices['seller_id'] == target_seller]
target_product_data = target_seller_data[target_seller_data['product_id'] == target_product]
